# Remaining experiments (3-7) after MC-dropout completed successfully.
Fixes model loading issue by rebuilding model from scratch.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import time
import json
import warnings
import os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Layer
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

results_dir = Path('../results')
tables_dir = results_dir / 'tables'

# Load data
print("Loading data...")
X_train_full = np.load(results_dir / 'X_train.npy')
X_test = np.load(results_dir / 'X_test.npy')
y_train_full = np.load(results_dir / 'y_train.npy')
y_test = np.load(results_dir / 'y_test.npy')

with open(results_dir / 'feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

val_size = int(0.2 * len(X_train_full))
X_val = X_train_full[-val_size:]
y_val = y_train_full[-val_size:]
X_train = X_train_full[:-val_size]
y_train = y_train_full[:-val_size]

print(f"Data loaded. X_train: {X_train.shape}, X_test: {X_test.shape}")

class AttentionLayer(Layer):
    def __init__(self, units=20, return_sequences=True, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.return_sequences = return_sequences
    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], self.units), initializer='glorot_uniform')
        self.b = self.add_weight(name='attention_bias', shape=(self.units,), initializer='zeros')
        self.u = self.add_weight(name='attention_vector', shape=(self.units,), initializer='glorot_uniform')
        super().build(input_shape)
    def call(self, inputs):
        score = tf.nn.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        aw = tf.nn.softmax(tf.tensordot(score, self.u, axes=1), axis=1)
        aw_exp = tf.expand_dims(aw, -1)
        ctx = tf.reduce_sum(inputs * aw_exp, axis=1)
        if self.return_sequences:
            return ctx, tf.squeeze(aw_exp, -1)
        return ctx
    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'return_sequences': self.return_sequences})
        return config

def build_attention_model(input_shape):
    inputs = Input(shape=input_shape)
    lstm_out = LSTM(20, return_sequences=True)(inputs)
    ctx, attn_w = AttentionLayer(units=20)(lstm_out)
    outputs = Dense(1)(ctx)
    model = Model(inputs=inputs, outputs=outputs)
    attn_model = Model(inputs=inputs, outputs=attn_w)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model, attn_model

# Train a fresh attention model for perturbation tests
print("\nTraining fresh attention-LSTM for perturbation experiments...")
np.random.seed(42)
tf.random.set_seed(42)
attn_model, attn_weights_model = build_attention_model((X_train.shape[1], X_train.shape[2]))
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
attn_model.fit(X_train, y_train, validation_data=(X_val, y_val),
               epochs=50, batch_size=256, callbacks=[es], verbose=0)
print("Training complete.")

# Load attention weights
attention_weights = np.load(results_dir / 'attention_weights.npy')
all_results = {}

## EXPERIMENT 3: ATTENTION PERTURBATION/ABLATION (R1-3)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 3: ATTENTION PERTURBATION/ABLATION (R1-3)")
print("="*70)

avg_attention = attention_weights.mean(axis=0)
timestep_order = np.argsort(avg_attention)[::-1]

n_perturb = min(50000, len(X_test))
np.random.seed(42)
perturb_idx = np.random.choice(len(X_test), n_perturb, replace=False)
X_perturb = X_test[perturb_idx].copy()
y_perturb = y_test[perturb_idx]

y_base = attn_model.predict(X_perturb, verbose=0).flatten()
base_mae = mean_absolute_error(y_perturb, y_base)
base_r2 = r2_score(y_perturb, y_base)

perturbation_results = [{'Perturbation': 'None (baseline)', 'MAE': base_mae, 'R2': base_r2, 'MAE_Increase_Pct': 0.0}]
print(f"Baseline: MAE={base_mae:.6f}, R²={base_r2:.4f}")

train_mean = X_train.mean(axis=0)

print("\n--- Mask high-attention timesteps ---")
for k in range(1, 6):
    X_m = X_perturb.copy()
    top_k = timestep_order[:k]
    for t in top_k:
        X_m[:, t, :] = train_mean[t, :]
    y_m = attn_model.predict(X_m, verbose=0).flatten()
    mae_m = mean_absolute_error(y_perturb, y_m)
    r2_m = r2_score(y_perturb, y_m)
    inc = ((mae_m - base_mae) / base_mae) * 100
    names = [f"t-{15-t}" for t in top_k]
    perturbation_results.append({'Perturbation': f'Mask top-{k} ({", ".join(names)})', 'MAE': mae_m, 'R2': r2_m, 'MAE_Increase_Pct': inc})
    print(f"  Mask top-{k} ({', '.join(names)}): MAE={mae_m:.6f} (+{inc:.1f}%), R²={r2_m:.4f}")

print("\n--- Mask low-attention timesteps ---")
for k in range(1, 6):
    X_m = X_perturb.copy()
    bot_k = timestep_order[-k:]
    for t in bot_k:
        X_m[:, t, :] = train_mean[t, :]
    y_m = attn_model.predict(X_m, verbose=0).flatten()
    mae_m = mean_absolute_error(y_perturb, y_m)
    r2_m = r2_score(y_perturb, y_m)
    inc = ((mae_m - base_mae) / base_mae) * 100
    names = [f"t-{15-t}" for t in bot_k]
    perturbation_results.append({'Perturbation': f'Mask bottom-{k} ({", ".join(names)})', 'MAE': mae_m, 'R2': r2_m, 'MAE_Increase_Pct': inc})
    print(f"  Mask bottom-{k} ({', '.join(names)}): MAE={mae_m:.6f} (+{inc:.1f}%), R²={r2_m:.4f}")

print("\n--- Random shuffle ---")
X_s = X_perturb.copy()
for i in range(len(X_s)):
    X_s[i] = X_s[i, np.random.permutation(15), :]
y_s = attn_model.predict(X_s, verbose=0).flatten()
mae_s = mean_absolute_error(y_perturb, y_s)
r2_s = r2_score(y_perturb, y_s)
inc_s = ((mae_s - base_mae) / base_mae) * 100
perturbation_results.append({'Perturbation': 'Random shuffle', 'MAE': mae_s, 'R2': r2_s, 'MAE_Increase_Pct': inc_s})
print(f"  Random shuffle: MAE={mae_s:.6f} (+{inc_s:.1f}%), R²={r2_s:.4f}")

pd.DataFrame(perturbation_results).to_csv(tables_dir / 'attention_perturbation_results.csv', index=False)
all_results['perturbation'] = perturbation_results
print("✅ Perturbation results saved")

## EXPERIMENT 4: HYPERPARAMETER SENSITIVITY (R1-7)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 4: HYPERPARAMETER SENSITIVITY ANALYSIS (R1-7)")
print("="*70)

hyp_sub = min(100000, len(X_train))
np.random.seed(42)
hyp_idx = np.random.choice(len(X_train), hyp_sub, replace=False)
X_hyp = X_train[hyp_idx]
y_hyp = y_train[hyp_idx]

configs = [
    {'lr': 0.001, 'dropout': 0.2, 'units': 20, 'label': 'Default (paper)'},
    {'lr': 0.0001, 'dropout': 0.2, 'units': 20, 'label': 'Lower LR (0.0001)'},
    {'lr': 0.01, 'dropout': 0.2, 'units': 20, 'label': 'Higher LR (0.01)'},
    {'lr': 0.001, 'dropout': 0.0, 'units': 20, 'label': 'No dropout'},
    {'lr': 0.001, 'dropout': 0.4, 'units': 20, 'label': 'Higher dropout (0.4)'},
    {'lr': 0.001, 'dropout': 0.2, 'units': 10, 'label': 'Fewer units (10)'},
    {'lr': 0.001, 'dropout': 0.2, 'units': 50, 'label': 'More units (50)'},
    {'lr': 0.001, 'dropout': 0.2, 'units': 100, 'label': 'Many units (100)'},
]
SEEDS_H = [42, 123, 456, 789]

hyperparam_results = []
for config in configs:
    print(f"\nConfig: {config['label']}")
    sm = {'rmse': [], 'mae': [], 'r2': []}
    conv = 0
    for seed in SEEDS_H:
        np.random.seed(seed); tf.random.set_seed(seed)
        m = Sequential([LSTM(config['units'], input_shape=(X_hyp.shape[1], X_hyp.shape[2])),
                        Dropout(config['dropout']), Dense(1)])
        m.compile(optimizer=keras.optimizers.Adam(learning_rate=config['lr']), loss='mse', metrics=['mae'])
        m.fit(X_hyp, y_hyp, validation_data=(X_val, y_val), epochs=50, batch_size=256,
              callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)], verbose=0)
        yp = m.predict(X_test, verbose=0).flatten()
        r = np.sqrt(mean_squared_error(y_test, yp)); ma = mean_absolute_error(y_test, yp); r2 = r2_score(y_test, yp)
        sm['rmse'].append(r); sm['mae'].append(ma); sm['r2'].append(r2)
        if r2 > -1.0: conv += 1
    res = {'Configuration': config['label'], 'LR': config['lr'], 'Dropout': config['dropout'],
           'Units': config['units'], 'Conv_Rate': f"{conv}/{len(SEEDS_H)}",
           'Mean_RMSE': np.mean(sm['rmse']), 'Std_RMSE': np.std(sm['rmse']),
           'Mean_MAE': np.mean(sm['mae']), 'Std_MAE': np.std(sm['mae']),
           'Mean_R2': np.mean(sm['r2']), 'Std_R2': np.std(sm['r2']),
           'Best_R2': max(sm['r2']), 'Worst_R2': min(sm['r2'])}
    hyperparam_results.append(res)
    print(f"  Conv: {conv}/{len(SEEDS_H)}, MAE: {res['Mean_MAE']:.4f}±{res['Std_MAE']:.4f}, R²: {res['Mean_R2']:.4f}±{res['Std_R2']:.4f}")

pd.DataFrame(hyperparam_results).to_csv(tables_dir / 'hyperparameter_sensitivity.csv', index=False)
all_results['hyperparameters'] = hyperparam_results
print("✅ Hyperparameter sensitivity saved")

## EXPERIMENT 5: INFERENCE LATENCY (R1-10)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 5: INFERENCE LATENCY ANALYSIS (R1-10)")
print("="*70)

# Build baseline model
np.random.seed(1415); tf.random.set_seed(1415)
baseline_m = Sequential([LSTM(20, input_shape=(X_train.shape[1], X_train.shape[2])), Dropout(0.2), Dense(1)])
baseline_m.compile(optimizer='adam', loss='mse')
baseline_m.fit(X_train[:100000], y_train[:100000], validation_data=(X_val, y_val),
               epochs=50, batch_size=256, callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)], verbose=0)

latency_results = []
batch_sizes = [1, 10, 100, 1000]
n_rep = 50

for bs in batch_sizes:
    X_b = X_test[:bs]
    # Warm up
    _ = baseline_m.predict(X_b, verbose=0)
    _ = attn_model.predict(X_b, verbose=0)

    tb = []
    for _ in range(n_rep):
        s = time.perf_counter(); _ = baseline_m.predict(X_b, verbose=0); tb.append(time.perf_counter()-s)
    ta = []
    for _ in range(n_rep):
        s = time.perf_counter(); _ = attn_model.predict(X_b, verbose=0); ta.append(time.perf_counter()-s)

    r = {'Batch_Size': bs, 'Baseline_ms': np.mean(tb)*1000, 'Baseline_std_ms': np.std(tb)*1000,
         'Attention_ms': np.mean(ta)*1000, 'Attention_std_ms': np.std(ta)*1000,
         'Per_sample_baseline_ms': np.mean(tb)*1000/bs, 'Per_sample_attention_ms': np.mean(ta)*1000/bs}
    latency_results.append(r)
    print(f"  Batch={bs}: Baseline={r['Baseline_ms']:.2f}ms, Attention={r['Attention_ms']:.2f}ms, Per-sample: {r['Per_sample_attention_ms']:.3f}ms")

# MC-dropout latency (100 passes, single sample)
X_single = X_test[:1]
mc_times = []
for _ in range(10):
    s = time.perf_counter()
    for _ in range(100):
        _ = attn_model(X_single, training=True)
    mc_times.append(time.perf_counter()-s)
print(f"  MC-dropout (100 passes, 1 sample): {np.mean(mc_times)*1000:.1f}ms")
latency_results.append({'Batch_Size': 'MC-100', 'Baseline_ms': 'N/A', 'Attention_ms': np.mean(mc_times)*1000})

pd.DataFrame(latency_results).to_csv(tables_dir / 'inference_latency.csv', index=False)
all_results['latency'] = latency_results
print("✅ Inference latency saved")

## EXPERIMENT 6: WINDOW SENSITIVITY (R1-12)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 6: WINDOW SENSITIVITY ANALYSIS (R1-12)")
print("="*70)

from sklearn.preprocessing import MinMaxScaler

data_path = Path('../github_repository/data/data.csv')
if not data_path.exists():
    data_path = Path('../../data.csv')
df_raw = pd.read_csv(str(data_path), encoding='latin-1')
fcols = ['humidity', 'temperature', 'humiditysol', 'temperaturesol', 'co2', 'lumière']
scaled = MinMaxScaler().fit_transform(df_raw[fcols].dropna().values)

window_sizes = [5, 10, 15, 20, 30]
SEEDS_W = [42, 123, 456]
window_results = []

for ws in window_sizes:
    print(f"\nWindow={ws} ({ws*5}min)")
    X_w = np.array([scaled[i-ws:i] for i in range(ws, len(scaled))])
    y_w = np.array([scaled[i, 2] for i in range(ws, len(scaled))])
    split = int(len(X_w)*0.8)
    X_tr_w, X_te_w = X_w[:split], X_w[split:]
    y_tr_w, y_te_w = y_w[:split], y_w[split:]

    sub = min(100000, len(X_tr_w))
    np.random.seed(42)
    si = np.random.choice(len(X_tr_w), sub, replace=False)
    vs = int(0.2*sub)

    sm = {'rmse': [], 'mae': [], 'r2': [], 'conv': 0}
    for seed in SEEDS_W:
        np.random.seed(seed); tf.random.set_seed(seed)
        inp = Input(shape=(ws, 6)); lo = LSTM(20, return_sequences=True)(inp)
        c, _ = AttentionLayer(20)(lo); out = Dense(1)(c)
        mw = Model(inputs=inp, outputs=out); mw.compile(optimizer='adam', loss='mse', metrics=['mae'])
        mw.fit(X_tr_w[si[:sub-vs]], y_tr_w[si[:sub-vs]], validation_data=(X_tr_w[si[sub-vs:]], y_tr_w[si[sub-vs:]]),
               epochs=50, batch_size=256, callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)], verbose=0)
        yp = mw.predict(X_te_w, verbose=0).flatten()
        sm['rmse'].append(np.sqrt(mean_squared_error(y_te_w, yp)))
        sm['mae'].append(mean_absolute_error(y_te_w, yp))
        sm['r2'].append(r2_score(y_te_w, yp))
        if sm['r2'][-1] > 0: sm['conv'] += 1

    r = {'Window': ws, 'Minutes': ws*5, 'Conv': f"{sm['conv']}/{len(SEEDS_W)}",
         'Mean_MAE': np.mean(sm['mae']), 'Std_MAE': np.std(sm['mae']),
         'Mean_R2': np.mean(sm['r2']), 'Std_R2': np.std(sm['r2'])}
    window_results.append(r)
    print(f"  Conv: {r['Conv']}, MAE: {r['Mean_MAE']:.4f}±{r['Std_MAE']:.4f}, R²: {r['Mean_R2']:.4f}±{r['Std_R2']:.4f}")

pd.DataFrame(window_results).to_csv(tables_dir / 'window_sensitivity.csv', index=False)
all_results['windows'] = window_results
print("✅ Window sensitivity saved")

## EXPERIMENT 7: EXTENDED SEEDS (R1-6)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 7: EXTENDED SEED ANALYSIS (n=16) (R1-6)")
print("="*70)

SEEDS_EXT = [42, 123, 456, 789, 1011, 1213, 1415, 1617,
             2020, 3030, 4040, 5050, 6060, 7070, 8080, 9090]

seed_sub = min(100000, len(X_train))
np.random.seed(42)
si = np.random.choice(len(X_train), seed_sub, replace=False)
vs = int(0.2*seed_sub)

ext_b, ext_a = [], []
for seed in SEEDS_EXT:
    # Baseline
    np.random.seed(seed); tf.random.set_seed(seed)
    mb = Sequential([LSTM(20, input_shape=(X_train.shape[1], X_train.shape[2])), Dropout(0.2), Dense(1)])
    mb.compile(optimizer='adam', loss='mse', metrics=['mae'])
    mb.fit(X_train[si[:seed_sub-vs]], y_train[si[:seed_sub-vs]],
           validation_data=(X_train[si[seed_sub-vs:]], y_train[si[seed_sub-vs:]]),
           epochs=50, batch_size=256, callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)], verbose=0)
    ypb = mb.predict(X_test, verbose=0).flatten()
    ext_b.append({'Seed': seed, 'RMSE': float(np.sqrt(mean_squared_error(y_test, ypb))),
                  'MAE': float(mean_absolute_error(y_test, ypb)), 'R2': float(r2_score(y_test, ypb))})

    # Attention
    np.random.seed(seed); tf.random.set_seed(seed)
    ma, _ = build_attention_model((X_train.shape[1], X_train.shape[2]))
    ma.fit(X_train[si[:seed_sub-vs]], y_train[si[:seed_sub-vs]],
           validation_data=(X_train[si[seed_sub-vs:]], y_train[si[seed_sub-vs:]]),
           epochs=50, batch_size=256, callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)], verbose=0)
    ypa = ma.predict(X_test, verbose=0).flatten()
    ext_a.append({'Seed': seed, 'RMSE': float(np.sqrt(mean_squared_error(y_test, ypa))),
                  'MAE': float(mean_absolute_error(y_test, ypa)), 'R2': float(r2_score(y_test, ypa))})

    print(f"  Seed {seed}: Base R²={ext_b[-1]['R2']:.4f}, Attn R²={ext_a[-1]['R2']:.4f}")

pd.DataFrame(ext_b).to_csv(tables_dir / 'extended_seed_baseline.csv', index=False)
pd.DataFrame(ext_a).to_csv(tables_dir / 'extended_seed_attention.csv', index=False)

bc = sum(1 for r in ext_b if r['R2'] > -1.0)
ac = sum(1 for r in ext_a if r['R2'] > 0)
print(f"\n--- Summary (n={len(SEEDS_EXT)}) ---")
print(f"  Baseline conv: {bc}/{len(SEEDS_EXT)} ({bc/len(SEEDS_EXT)*100:.1f}%)")
print(f"  Attention conv: {ac}/{len(SEEDS_EXT)} ({ac/len(SEEDS_EXT)*100:.1f}%)")
print(f"  Baseline R²: mean={np.mean([r['R2'] for r in ext_b]):.2f}")
print(f"  Attention R²: mean={np.mean([r['R2'] for r in ext_a]):.4f} ± {np.std([r['R2'] for r in ext_a]):.4f}")

all_results['extended_seeds'] = {'baseline': ext_b, 'attention': ext_a}
print("✅ Extended seed analysis complete")

# Save all
with open(tables_dir / 'reviewer_experiment_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print("\n✅ ALL EXPERIMENTS COMPLETE")